In [ ]:
!sudo apt update
!sudo apt install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [15]:
!ollama pull mxbai-embed-large


In [16]:
!ollama pull llama3.2:3b

In [17]:
!ollama list

NAME                        ID              SIZE      MODIFIED       
llama3.2:3b                 a80c4f17acd5    2.0 GB    1 second ago      
mxbai-embed-large:latest    468836162de7    669 MB    33 seconds ago    


In [ ]:
!pip install langchain_community chromadb pypdf

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter

In [7]:
def docs_preprocessing_helper(file):
  loader = PyPDFLoader(file)
  docs = loader.load_and_split()

  text_splitter = CharacterTextSplitter(chunk_size=5000, chunk_overlap=800)
  docs = text_splitter.split_documents(docs)

  return docs

In [8]:
docs = docs_preprocessing_helper('./machine-learning-engineer-associate-01.pdf')

In [ ]:
!pip install langchain_ollama

In [10]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="llama3.2:3b", temperature=0.0)

In [11]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="mxbai-embed-large:latest",
)

In [ ]:
!pip install opentelemetry-api opentelemetry-sdk

In [13]:
import chromadb
from langchain_community.vectorstores import Chroma

In [18]:
db = Chroma.from_documents(docs, embeddings, persist_directory="my_chroma_db")
db.persist()


/tmp/ipykernel_23078/3335644042.py:2: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()


In [20]:
import langchain
from langchain_core.prompts import PromptTemplate

# =============================================================================
# 5. Define and Initialize the Prompt Template
# =============================================================================

# Define a prompt template that instructs the chatbot on how to answer queries.
# The template includes context information and instructs the bot to use only provided data.
template = """You are a teaching chatbot. Use only the source data provided to answer.

If the answer is not in the source data or is incomplete, say:
"I’m sorry, but I couldn’t find the information in the provided data."

{context}

"""

# Create a PromptTemplate object from LangChain with the defined template.
# It expects a variable called "context" that can be filled later.
prompt = PromptTemplate(template=template, input_variables=["context"])

# Format the prompt with a general context message.
# This additional context tells the chatbot the scenario in which it will be answering questions.
formatted_prompt = prompt.format(
    context="You are interacting with college students. They will ask you questions related to the file provided. Please answer their specific questions using the provided file."
)

# Define a refine prompt for iterative refinement if new context is provided.
refine_prompt_template = """You are a teaching chatbot. We have an existing answer:
{existing_answer}

We have the following new context to consider:
{context}

Please refine the original answer if there's new or better information.
If the new context does not change or add anything to the original answer, keep it the same.

If the answer is not in the source data or is incomplete, say:
"I’m sorry, but I couldn’t find the information in the provided data."

Question: {question}

Refined Answer:
"""

refine_prompt = PromptTemplate(
    template=refine_prompt_template,
    input_variables=["existing_answer", "context", "question"]
)

In [21]:
from langchain_classic.chains import RetrievalQA

# =============================================================================
# 6. Create the RetrievalQA Chain
# =============================================================================

# The RetrievalQA chain combines:
#   - The language model (model) to generate responses.
#   - A retriever (db.as_retriever) that fetches relevant document chunks based on the query.
#   - A prompt that provides instructions on how to answer the query.
chain_type_kwargs = {
    "question_prompt": prompt,
    "refine_prompt": refine_prompt,
    "document_variable_name": "context",
}

chain = RetrievalQA.from_chain_type(
    llm=model,
    chain_type="refine",  # "refine" iteratively improves the answer based on additional context.
    retriever=db.as_retriever(search_kwargs={"k": 5}),  # Retrieve the top 5 relevant documents.
    chain_type_kwargs=chain_type_kwargs,
)


In [22]:

# =============================================================================
# 7. Query the Chain and Output the Response
# =============================================================================

# Define a query related to the PDF content.
query = "What are hyperparameter tuning techniques?"

# Run the chain with the query. The chain will:
#   1. Retrieve the most relevant document chunk(s) from ChromaDB.
#   2. Format the prompt with that context.
#   3. Use the language model to generate an answer based on the prompt and retrieved data.
response = chain.run(query)

# Print the response to the console.
print(response)

/tmp/ipykernel_23078/3798837281.py:12: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  response = chain.run(query)


Hyperparameter tuning techniques refer to methods used to optimize the performance of machine learning models by adjusting the values of their hyperparameters. Hyperparameters are parameters that are set before training a model and cannot be adjusted during training, whereas model parameters are learned during training.

Some common hyperparameter tuning techniques include:

1. **Random Search**: This involves randomly sampling from a distribution over the possible values of hyperparameters and evaluating the performance of the model for each combination.
2. **Grid Search**: Similar to random search, but instead of sampling from a distribution, all possible combinations of hyperparameters are evaluated exactly once.
3. **Bayesian Optimization**: This uses a probabilistic approach to search for the optimal hyperparameters by modeling the relationship between hyperparameters and performance using a Bayesian model.
4. **Gradient-Based Optimization**: This involves using gradient-based met